# STEP 1: Dataset selection

**Dataset Name:**
WikiANN (English)

**Label Categories:**

O → Outside any entity  
B-PER → Beginning of person  
I-PER → Inside person  
B-ORG → Beginning of organization  
I-ORG → Inside organization  
B-LOC → Beginning of location  
I-LOC → Inside location  


In [ ]:
#Install libraries
!pip install transformers datasets seqeval

In [ ]:
#Load Dataset
from datasets import load_dataset
dataset = load_dataset("wikiann", "en")
dataset

In [ ]:
#Check Labels
label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)

Since CoNLL-2003 and UD were unavailable, WikiANN dataset was used. It follows BIO tagging, similar to chunking, making it suitable for token classification.

# STEP 2: Tokenization + Label Alignment

In [ ]:
#Load Tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
#Define Label Alignment Function
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Special tokens
            if word_idx is None:
                label_ids.append(-100)

            # First token of word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            # Subword tokens
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
#Apply to Dataset
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
#Check output
print(tokenized_dataset["train"][0])

**Data Preprocessing:**

Used BERT tokenizer

Handled subword tokenization

Assigned -100 to special tokens and subwords

Aligned labels using word_ids()

# STEP 3: Model Setup

In [ ]:
#Define label mappings
label_list = dataset["train"].features["ner_tags"].feature.names

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

print(label2id)

In [ ]:
#Load Model
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

In [ ]:
#Verify Model
print(model.config)

**Model Setup:**

Used BERT (bert-base-uncased)

Used AutoModelForTokenClassification

Defined:
num_labels,
label2id,
id2label

# STEP 4: Training

In [ ]:
#Install
!pip install accelerate

In [ ]:
#Import
from transformers import TrainingArguments, Trainer


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
)

In [ ]:
#Define Data Collator
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
!pip install evaluate

In [ ]:
#Load Metric
import evaluate

metric = evaluate.load("seqeval")

In [ ]:
#Define compute_metrics
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
#Train Model
trainer.train()

**Training:**

Used Hugging Face Trainer API

Learning rate: 2e-5

Epochs: 2

Batch size: 16

Used seqeval for evaluation

# STEP 5: Evaluation

In [ ]:
#Run Evaluation
results = trainer.evaluate()
print(results)

**Evaluation:**

Used seqeval metric

Evaluated model on validation dataset

Metrics used:
Precision,
Recall,
F1 Score

# STEP 6: Inference

In [ ]:
#Load pipeline
from transformers import pipeline

ner_pipeline = pipeline("token-classification", model=model, tokenizer=tokenizer)

In [ ]:
#Test on custom sentence
sentence = "John works at Google in California"

predictions = ner_pipeline(sentence)

for pred in predictions:
    print(pred)

In [ ]:
for pred in predictions:
    print(f"{pred['word']} → {pred['entity']}")

**Inference:**

Used trained model for prediction

Tested on custom sentence

Model correctly identified entities like Person, Organization, Location

# STEP 7: Comparison

***Comparison: POS Tagging vs Chunking vs NER***

**1. POS Tagging**

Assigns grammatical labels to each word

Example: Noun (NN), Verb (VB), Adjective (JJ)

Works at word level

Easier task

**2. Chunking (Shallow Parsing)**

Groups words into meaningful phrases

Example: Noun Phrase (NP), Verb Phrase (VP)

Uses POS tags to form phrases

Works at phrase level

Moderate difficulty

**3. NER **


Identifies named entities like:
Person,
Location,
Organization

Uses BIO tagging (B-, I-, O)

Similar to chunking but focuses on real-world entities

*POS tagging works at word level, chunking groups words into phrases, and NER identifies real-world entities using BIO tagging.*

**Introduction**

For this particular project, a model using transformers (BERT) has been fine-tuned for token classification purposes. However, due to unavailability of datasets, the model uses the WikiANN dataset for Named Entity Recognition (NER), similar to chunking where BIO tagging technique is employed.

**Differences**

POS Tagging uses grammatical tags.

Chunking classifies words according to phrases such as noun phrases or verb phrases.

NER classifies real-life entities such as names, places, or organizations.

**Challenges Encountered**

Sub-word tokenization problem

Matching tokens and labels

Using special tokens with -100

Library compatibility problems

**Findings**

BERT is a suitable model for token classification tasks

Label matching is the most crucial task

NER offers real-world insight in comparison to POS tagging